# SubROC: where does a classifier perform poorly?

Subgroup discovery can also audit a model: instead of describing where a target
is unusual, it finds descriptions of the data where the **model performs much
worse** than overall. We train scikit-learn's gradient boosting classifier on
the Adult census data (it handles the missing values and categorical columns
directly) and search the **test set** for subgroups where its ranking quality,
the ROC AUC, is much lower than overall.

In [ ]:
from sklearn.datasets import fetch_openml

# Adult / Census Income (OpenML dataset 1590): 48,842 people, income above 50K or not.
adult = fetch_openml("adult", version=2, as_frame=True)
# fnlwgt is a census sampling weight, not a property of the person.
data = adult.frame.drop(columns=["fnlwgt"])
features = data.drop(columns="class")

## Train a classifier

The model predicts whether a person earns more than 50K; we keep its scores on
the held-out test set.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features, data["class"], test_size=0.3, stratify=data["class"], random_state=0
)
model = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=0)
model.fit(X_train, y_train)
scores = model.predict_proba(X_test)[:, list(model.classes_).index(">50K")]
audit = X_test.assign(label=(y_test == ">50K").to_numpy(), score=scores).reset_index(drop=True)
print(f"test ROC AUC: {roc_auc_score(audit.label, audit.score):.3f}")

## Search for subgroups with a low AUC

`SoftClassifierTarget` stores the true labels and the model's scores;
`ROCAUCQF` scores the AUC decrease (the SubROC measures, Siegl et al.).
The descriptions use the features only, not the label or the score.

In [ ]:
import pysubgroup as sd

performance = sd.SoftClassifierTarget().fit(audit, "label", "score", positive_value=True)
task = sd.SubgroupDiscoveryTask(
    audit, performance, sd.generate_conditions(X_test, n_bins=5), sd.ROCAUCQF(),
    result_set_size=5, depth=2, filters=[sd.MinSizeFilter(300)],
)
found = sd.DFS().execute(task)
table = sd.to_dataframe(found, audit, performance, ["size", "target_share", "roc_auc"])
table.rename(columns={"quality": "AUC decrease", "roc_auc": "subgroup AUC"}).round(3)

## The full SubROC framework

The search above favors small subgroups with an extreme class balance, whose
low AUC is often chance. SubROC (Siegl et al., Sect. 3) adds three remedies:

- **Weighting** by size and class balance:
  `ROCAUCQF(subgroup_class_balance_weight=1, subgroup_size_weight=1)`.
- **Generalization awareness**: a refinement scores only its gain over all its
  generalizations (`GeneralizationAwareQF`, searched by `GeneralizationAwareApriori`).
- **Significance filtering on a holdout set**: search the top-k' subgroups
  (here 20) on one half of the data, test each on the other half, and keep the
  top-k (here 5) significant ones.

We first run the significance filtering step by step and then with the single
call `holdout_search`.

In [ ]:
import numpy as np

qf = sd.GeneralizationAwareQF(
    sd.ROCAUCQF(subgroup_class_balance_weight=1, subgroup_size_weight=1)
)
conditions = sd.generate_conditions(X_test, n_bins=5)

### Step by step

**1. Split** the audited test set into a search half and a validation half,
stratified by the label. Each half gets its own target, so the dataset AUC is
that of the half. One random generator drives the split and the test below.

In [ ]:
rng = np.random.default_rng(0)
search, validation = sd.holdout_split(audit, 0.5, stratify="label", random_state=rng)
search_target = sd.SoftClassifierTarget().fit(search, "label", "score", positive_value=True)
validation_target = sd.SoftClassifierTarget().fit(
    validation, "label", "score", positive_value=True
)
print(f"search: {len(search)} rows, validation: {len(validation)} rows")

**2. Search** the top-k' = 20 subgroups on the search half.

In [ ]:
search_task = sd.SubgroupDiscoveryTask(
    search, search_target, conditions, qf,
    result_set_size=20, depth=2, filters=[sd.MinSizeFilter(100)],
)
candidates = sd.GeneralizationAwareApriori().execute(search_task)

**3. Test** every candidate on the validation half. The randomization test
draws random subsets with the candidate's size and, with `stratify_target=True`
as in the paper, its class balance (by default only the size is kept). It
counts how often they score at least as high (the p-value, as in the paper,
with 1,000 random subsets); Benjamini-Yekutieli corrects for the 20 tests.

In [ ]:
p_values = sd.randomization_p_values(
    candidates, validation, validation_target, qf, n_samples=1000,
    stratify_target=True, random_state=rng,
)
adjusted = sd.adjust_p_values(p_values, "by")
significant = adjusted <= 0.05
print(f"{significant.sum()} of {len(candidates)} candidates are significant")

The test can be inspected: `plot_null_distribution` draws each candidate's
statistic (dot) over the statistics of its random subsets (box: quartiles,
whiskers: 2.5th to 97.5th percentiles), with the uncorrected p-value. Here with
300 random subsets per candidate; its own random state leaves `rng`, and
thus the comparison with `holdout_search` below, unchanged.

In [ ]:
from pysubgroup import visualization as viz

viz.plot_null_distribution(
    candidates, validation, validation_target, qf, n_samples=300,
    stratify_target=True, random_state=1,
);

**4. Keep** the top-k = 5 significant candidates and show all candidates with
their search-set statistics and p-values.

In [ ]:
results = [candidate for candidate, keep in zip(candidates, significant, strict=True) if keep][:5]
table = sd.to_dataframe(candidates, search, search_target, ["size", "target_share", "roc_auc"])
table.assign(p_value=p_values, adjusted_p_value=adjusted, significant=significant).round(4)

The ROC curves of the selected subgroups on the validation half, against the
whole validation half:

In [ ]:
viz.plot_performance_curves(results, validation, validation_target);

### One call

`holdout_search` performs the four steps. The task describes the search on the
whole audited test set with `result_set_size` = k'; the target is refitted on
both halves. With the same seed it gives the same subgroups as above.

In [ ]:
task = sd.SubgroupDiscoveryTask(
    audit, performance, conditions, qf,
    result_set_size=20, depth=2, filters=[sd.MinSizeFilter(100)],
)
holdout = sd.holdout_search(
    sd.GeneralizationAwareApriori(), task, k=5, split_stratify="label",
    n_samples=1000, stratify_target=True, random_state=0,
)
print("same subgroups as step by step:", holdout.results == results)

`to_dataframe` shows the selected subgroups with their statistics on the search
half, the same statistics on the validation half, and the p-values
(`candidates=True` shows all 20):

In [ ]:
holdout.to_dataframe(["size", "target_share", "roc_auc"]).round(4)

In [ ]:
viz.plot_performance_curves(
    holdout.results, holdout.validation_data, holdout.validation_target, curve="pr"
);